# Week 3, day 1 (afternoon) — Worksheet 09 SOLUTIONS: apply and map   (L02)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Question 4 is the one to re-read. `map` silently discards a value the lookup
does not cover, and the deck's example is constructed so this never shows.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 09 — apply and map. Run this once.
import pandas as pd

# The deck's example, plus one student it does not account for.
grades = pd.DataFrame(
    {"Grade": ["A", "B", "C", "D"]},
    index=["S1", "S2", "S3", "S4"],
)
grade_map = {"A": "Excellent", "B": "Good", "C": "Needs Review"}

marks = pd.Series([85, 90, 55, 72], index=["S1", "S2", "S3", "S4"], name="Marks")

sales = pd.read_csv("data/sales.csv")

print(grades)
print()
print("grade_map covers:", list(grade_map))

PART A — apply

### Question 1

`marks.apply(pass_fail)` -> `Pass, Pass, Fail, Pass`, `dtype: str`.

`apply` called your function once per value and collected the returns into
a new Series, keeping the index. S3 scored 55 and is the only Fail.

The result dtype is `str` because every return value was a string —
Pandas inferred it from what came back, not from anything you declared.

In [ ]:
def pass_fail(mark):
    return "Pass" if mark >= 60 else "Fail"

print(marks.apply(pass_fail))

### Question 2

The lambda gives the identical result; `.equals()` is `True`.

A lambda is the same function without a name. Prefer the named version
when the logic is worth explaining or reusing, and the lambda when it is a
one-line detail that would be noise as a definition.

The threshold `60` is buried in both. In real code that belongs in a named
constant, because it is a policy decision and someone will eventually ask
where it came from.

In [ ]:
by_lambda = marks.apply(lambda m: "Pass" if m >= 60 else "Fail")

def pass_fail(mark):
    return "Pass" if mark >= 60 else "Fail"

print(by_lambda)
print()
print("matches the named version:", by_lambda.equals(marks.apply(pass_fail)))

### Question 3

`marks >= 60` -> a boolean Series. `.ge(60).map({True: "Pass", False: "Fail"})` -> the same words as Q1.

Three spellings of one rule, and they differ in cost rather than result.

`apply` runs a Python function once per row — four calls here, four
million on a four-million-row frame, each with the full overhead of a
Python call. The comparison `marks >= 60` does the same work in a single
vectorised operation that never enters the interpreter per element.

On small data this is invisible. On large data it is the difference
between a second and several minutes. The rule of thumb: if what you are
doing can be expressed with operators and built-in methods, do that;
reserve `apply` for logic that genuinely cannot be vectorised.

In [ ]:
print("the mask itself:")
print(marks >= 60)
print()
print("mapped to words:")
print(marks.ge(60).map({True: "Pass", False: "Fail"}))

PART B — map, and the value nobody planned for

### Question 4

`map(grade_map)` -> `Excellent, Good, Needs Review`, and **`S4` is `NaN`**. -> nothing raised, nothing warned.

`D` is not a key in `grade_map`, so `map` produced a missing value for it
and said nothing.

The deck's version of this slide has exactly three students holding
exactly the three grades the dictionary defines. Nothing can go wrong in
it, so the behaviour never appears — and this is the single most common way
data quietly disappears in a Pandas pipeline. A new category shows up
upstream, your lookup does not know it, and the rows become `NaN` without
a single line of output changing.

Note the dtype is still `str`; a text column holds `NaN` without
promotion, so even the dtype gives you no hint.

In [ ]:
out = grades["Grade"].map(grade_map)
print(out)
print()
print("NaN count:", out.isna().sum())
print("nothing raised, nothing warned")

### Question 5

present `['A','B','C','D']`, covered `['A','B','C']`, **uncovered `['D']`**.

One line, and it is the check that would have caught Q4 before it
happened. Set difference between the values in the data and the keys in
the lookup tells you exactly which categories will be lost.

Worth running every time you map a column through a hand-written
dictionary — especially on a schedule, because the failure mode is a new
value appearing months after the code was written and reviewed.

In [ ]:
present = set(grades["Grade"])
covered = set(grade_map)
print("present in data:", sorted(present))
print("covered by map: ", sorted(covered))
print("UNCOVERED:      ", sorted(present - covered))

### Question 6

`.fillna("Unknown")` and `.map(lambda g: grade_map.get(g, "Unknown"))` -> identical results, `S4` becomes `Unknown`.

Same output, different guarantees. `fillna` repairs the damage after the
fact and cannot distinguish between 'this grade was not in the lookup' and
'this grade was already missing in the source data' — both become
`Unknown`. `.get(g, "Unknown")` never creates the `NaN`, so a genuinely
missing input stays `NaN` and only the unmapped ones become `Unknown`.

If you need to tell those two cases apart later — and on real data you
usually do — the second form preserves the distinction.

In [ ]:
a = grades["Grade"].map(grade_map).fillna("Unknown")
b = grades["Grade"].map(lambda g: grade_map.get(g, "Unknown"))
print(a)
print()
print("agree:", a.equals(b))

# fillna repairs the damage afterwards; .get() prevents it. Prefer the
# second when you know a default is acceptable, because it never creates
# the NaN in the first place.

PART C — on the real file

### Question 7

`Band` -> `small 241`, `large 59`, totalling `300`.

The 59 matches worksheet 07 Q1 exactly, which is the point of checking:
the same threshold applied through a different mechanism gives the same
answer.

The two counts add to 300 because the lambda returns a string on every
path — there is no branch that returns `None` and no value that fails the
comparison. When a derived column's `value_counts()` does not add up to
the row count, an unhandled case is the first thing to look for.

In [ ]:
sales["Band"] = sales["Sales"].apply(lambda v: "large" if v > 1000 else "small")
counts = sales["Band"].value_counts()
print(counts)
print()
print("total:", counts.sum(), "of", len(sales))

### Question 8

`value_counts(dropna=False)` -> **`NaN 150`**, `Central 80`, `Pacific 70`. -> mapped `150`, unmapped `150`.

Exactly half the file failed to map, and the lookup covered only the two
largest regions.

Without `dropna=False` the tally would read `Central 80, Pacific 70` and
total 150 — a clean-looking summary of a 300-row file with no indication
that half of it is missing. The rows are not merely uncounted; they are
invisible.

This is the practical reason to make `dropna=False` your default when
checking a derived column. The missing count is usually the most
informative number in the table.

In [ ]:
partial = {"Ontario": "Central", "West": "Pacific"}
zone = sales["Region"].map(partial)
print(zone.value_counts(dropna=False))
print()
print("mapped:   ", zone.notna().sum())
print("unmapped: ", zone.isna().sum())
print("total:    ", len(zone))

### Question 9

`apply(..., axis=1)` -> `143.020, 2763.149, 975.310, 133.330, 154.390`. -> identical to `sales["Sales"] - sales["Profit"]`.

`axis=1` hands your function one row at a time as a Series, so `r["Sales"]`
works. Without it, `apply` passes whole columns and `r["Sales"]` is a
lookup by row label into a column — an error, or worse, an accidental
success.

The two results are identical and the vectorised form is the one to write.
Row-wise `apply` is the slowest thing in this sheet by a wide margin,
because it constructs a Series object for every single row. Use it only
when the logic genuinely needs several columns at once in a way arithmetic
cannot express.

In [ ]:
by_row = sales.apply(lambda r: r["Sales"] - r["Profit"], axis=1)
vectorised = sales["Sales"] - sales["Profit"]

print(by_row.head())
print()
print("identical to the vectorised form:", by_row.equals(vectorised))

### Question 10

`marks.apply(lambda m: 100 / (m - 72))` -> **raises** `ZeroDivisionError: division by zero`.

S4 scored exactly 72, so the denominator is zero and the whole operation
stops. You get no partial result — the three values that would have
computed fine are lost with it.

Put that beside Q4 and you have the contrast the sheet is built around:

- **`map`** met a value its lookup did not cover and returned `NaN` for it,
  silently, keeping every other row.
- **`apply`** met a value its function could not handle and raised,
  discarding everything.

Neither is better. The loud one costs you a run and tells you exactly
where the problem is; the quiet one costs you nothing now and may cost you
a wrong report later. Knowing which of the two you are using — and
therefore which failure mode you have signed up for — is the actual skill.

In [ ]:
print(marks.apply(lambda m: 100 / (m - 72)))